Focus: Problem formulation, environment, training, algorithm, wrappers, observation/action space, reward design, training pipeline.

## Mingrid - Gaming RL Project

### Problem Definition

Minigrid is game that that the goal is to navigate a square grid avoid optiscals and reach a exit to the next map. Obsticals is mad up of lava and walls. The levels are genrated by a random generator in training and the model is tested on a set of premade levels.

The main problem questioning is: Can an RL agent trained on randomly generated MiniGrid environments generalize well enough to solve previously unseen fixed levels?

Rules defiend by the game:
-   Levels: In testing is 6x6, 7x7, 9x9, 11x11 and 16x16 grids. The training levels are 10x10 grids
-   Charecter movement: Left, Right and Forward. Each action counting as a step (add in that we limited the character to this)
-   Obsticals: Walls limit movement and lava ends game with negative outcome
-   Field of view: The grid of view that the charecter sees is a 7x7 grid, where the chrecter is placed in the middel of a edge of the grid. Only seeing forward and to the sides
-   Goal: Is to reach green square that will move the charecter to next level


Problems for the model to solve is:
-   How to move the character, learn the inputs it can make
-   Learn how those movement inputs affect the reward
-   Use its field of view to identify the objects in that that space
-   Find a strategy move through the level 
-   With the overarcing goal to reach the exit as fast as possible

The value for evaluate how well the agent been able to genrilize to solve unseen levels is !!Fråga vad för metric vi har för preformence!! 
succes rate where we run the agent limited number runs to solve the level. 

### Limitaions

The agent action space is limitid to only movment. There is more action given by the `MiniGridEnv`, like pick up key or unlock door. This is due to not having thoe objects present in the train or test levels. Training is limtied to use 10x10 leval grids that is random genrated. Where the test set is fixed set of 5 levels made from 6x6, 7x7, 9x9, 11x11 and 16x16 grids. 

Reward function and observatin space is predefiende by `MiniGridEnv`.


### Reward function

The reward function is inherited directly from `MiniGridEnv`. This function only rewards the agent for successfuly reaching the goal, the green tile in the grid. 
The terminal reward is time-scaled to the number of steps taken in the level by the agant in each episod. It is defiend by:

$$
reward = 1 - \frac{steps\_taken}{max\_steps}
$$

Where `steps_taken` is number of moves in the enviroment made by the agent and `max_step` is the  max limite steps to be made. By this reward defintion the agent gets higher reward for succsefully completing the level as fast as possible, least amount of steps taken witin a episod. When failing to complet the level the reward is zero. The reward function is strictly spare due to no intermediate reawards,  

## Agent Structure

The agent interacts with the environment through its **observation space**, **action space**, and **reward function**. The reward function is as previously defined. The agent uses `CnnPolicy` from Stable Baselines3, which integrates a feature extractor (CNN) with separate policy and value networks and is trained using PPO. This class automates the mapping from image observations to actions and value estimates.

Observations are provided as **image-based tensors** via the `ImgObsWrapper`, with a grid size of 7x7. Each tile in the observation space is encoded as a tuple of 3 integers representing **object type, color, and object state**. This forms a **spatial representation** of the environment, which is processed by the CNN as an image.

The agent is implemented using the **PPO (Proximal Policy Optimization)** algorithm, an on-policy reinforcement learning method. The overall architecture consists of:

1. **Observation input:** 7x7x3 image from the MiniGrid environment.
2. **Feature extractor:** Custom CNN (`MinigridFeaturesExtractor`) converts the image into a 128-dimensional feature vector.
3. **Policy network (actor):** Fully connected layers map CNN features to probabilities over three discrete actions (left, right, forward). This network determines the agent's actions.
4. **Value network (critic):** Fully connected layers map CNN features to a scalar value estimating expected return. Used internally for PPO updates.
5. **PPO algorithm:** Trains the policy and value networks using advantage estimation, clipped policy updates, and stochastic sampling. This ensures stable learning from the image-based observations.

### Feature Extraction Network (CNN)

The CNN processes the image observations and extracts spatial features for decision-making:

- **Input:** 7x7x3 image tensor
- **Conv2D layer 1:** 16 filters, kernel size 2x2, activation `ReLU`
- **Conv2D layer 2:** 32 filters, kernel size 2x2, activation `ReLU`
- **Conv2D layer 3:** 64 filters, kernel size 2x2, activation `ReLU`
- **Flatten:** Converts 3D feature map to 1D vector
- **Fully connected layer:** Output dimension = 128, activation `ReLU`

**Output:** 128-dimensional latent feature vector representing the environment state.

### Policy and Value Networks

On top of the CNN features, PPO maintains separate networks:

- **Policy network:** Maps the 128-dimensional features to a probability distribution over actions. Determines the agent's behavior.
- **Value network:** Maps the features to a scalar value estimating expected return, used for training stability.

Both networks share the CNN features but have separate final weights, allowing the agent to learn effective policies while estimating state values independently.


### Training Process

The agent is trained using **Proximal Policy Optimization (PPO)** in combination with a CNN-based feature extractor (`MinigridFeaturesExtractor`). Training is designed to improve the policy (actor) and value (critic) networks so that the agent can solve the MiniGrid levels efficiently. 

The training process involves the following steps:

1. **Parallel Environments:**  
   - Multiple instances of the environment are created using `EnvOptimiser` and wrapped with `ImgObsWrapper`.  
   - Each environment runs independently in parallel, allowing the agent to collect diverse experiences simultaneously.  
   - This increases sample efficiency and stabilizes training.

2. **Interaction and Data Collection:**  
   - At each timestep, the agent observes the environment as a 7x7x3 image.  
   - The CNN converts the observation into a 128-dimensional feature vector.  
   - The **policy network** samples an action based on these features.  
   - The environment executes the action and returns the **next observation, reward, and termination info**.  
   - These interactions (state, action, reward, next state) are stored as trajectories for learning.

3. **Advantage Estimation and Network Updates:**  
   - PPO computes **advantages**, which measure how much better an action performed compared to the expected value.  
   - The **policy network** is updated to increase the probability of better-than-expected actions, using **clipped gradient updates** to prevent overly large changes.  
   - The **value network** is updated to reduce the difference between predicted and actual returns.  
   - Both networks are trained together using the collected trajectories.

4. **Progress Monitoring and Checkpointing:**  
   - During training, the callback `SaveOnBestTrainingRewardCallback` monitors the agent's mean reward over recent episodes.  
   - The model achieving the highest mean reward is automatically saved for evaluation.

Overall, the training process allows the agent to **learn from experience** by gradually improving its policy and value estimations, ultimately enabling it to navigate the environment and reach the goal efficiently.


### Methodology

Purpose: Explain how learning is performed.

5.1 Algorithm

RL algorithm used (e.g., PPO, DQN)

Why this algorithm was chosen

On-policy vs off-policy considerations

5.2 Model Architecture

Policy network structure

CNN vs MLP

Input processing (image resizing, normalization)

Value function (if applicable)

Parameter count (optional)

5.3 Observation and Action Wrappers

Wrappers applied (e.g., ImgObsWrapper, frame stacking)

Motivation for each wrapper

Impact on observation space

### Training and interference